# NeRSemble -> WanVideoVAE demo

Loads a single NeRSemble sequence, selects the 8 upper cameras, applies color correction, and runs encode/decode with `DiffSynth-Studio/diffsynth/models/wan_video_vae.py`.


In [ ]:
from pathlib import Path
import json
import re
import sys
import types

import numpy as np
import torch
import matplotlib.pyplot as plt

workspace_root = Path("/home/piado/projects/aip-lindell/piado")
nersemble_root = Path("/scratch/piado/data/nersemble")
diffsynth_root = workspace_root / "DiffSynth-Studio"
nersemble_pkg_root = workspace_root / "nersemble-data" / "src"

# Local imports for the downloaded dataset.
sys.path.insert(0, str(nersemble_pkg_root))
sys.path.insert(0, str(diffsynth_root))


In [ ]:
from nersemble_data.data.nersemble_data import (
    NeRSembleDataManager,
    NeRSembleParticipantDataManager,
)

data_folder = NeRSembleDataManager(str(nersemble_root))
participants = sorted(data_folder.list_participants())
if not participants:
    raise RuntimeError("No participants found in the NeRSemble folder.")

# Prefer participant 017 if available; otherwise use the first.
participant_id = 17 if 17 in participants else participants[0]
data_manager = NeRSembleParticipantDataManager(str(nersemble_root), participant_id)

sequences = data_manager.list_sequences()
if not sequences:
    raise RuntimeError(f"No sequences found for participant {participant_id}.")

# Prefer a non-background sequence for reconstruction.
sequence_name = next((s for s in sequences if s != "BACKGROUND"), sequences[0])

print("Participant:", participant_id)
print("Sequence:", sequence_name)
print("Available sequences:", sequences)


In [ ]:
# Load camera calibration via the data manager.
camera_calibration = data_manager.load_camera_calibration()
world_2_cam_poses = camera_calibration.world_2_cam
intrinsics = camera_calibration.intrinsics

print("Intrinsics:\n", intrinsics)
print("Number of cameras in calibration:", len(world_2_cam_poses))


In [ ]:
def pick_upper_cameras(
    root: Path,
    participant_id: int,
    sequence_name: str,
    top_k: int = 8,
    axis: int = 1,
):
    """
    Select upper cameras by sorting camera centers along a world-axis.
    Assumes world axis 1 (Y) is "up" for this dataset.
    """
    images_dir = root / f"{participant_id:03d}/sequences/{sequence_name}/images"
    if images_dir.exists():
        available = [p.stem.split("_")[1] for p in images_dir.glob("cam_*.mp4")]
    else:
        available = data_manager.list_cameras(sequence_name)

    cam_json = root / f"{participant_id:03d}/calibration/camera_params.json"
    world_2_cam = json.loads(cam_json.read_text())["world_2_cam"]

    centers = {}
    for serial in available:
        w2c = np.array(world_2_cam[serial], dtype=np.float32)
        c2w = np.linalg.inv(w2c)
        centers[serial] = c2w[:3, 3]

    ordered = sorted(available, key=lambda s: centers[s][axis], reverse=True)
    return ordered[:top_k], centers


try:
    upper_serials, camera_centers = pick_upper_cameras(
        nersemble_root, participant_id, sequence_name, top_k=8, axis=1
    )
except Exception as exc:
    print("Upper camera selection failed, falling back to sorted cameras:", exc)
    upper_serials = sorted(data_manager.list_cameras(sequence_name))[:8]

print("Upper-view cameras (8):", upper_serials)


In [ ]:
timestep = 0
downscale_factor = 4  # set to None for full resolution

images = []
for serial in upper_serials:
    img = data_manager.load_image(
        sequence_name,
        serial,
        timestep,
        apply_color_correction=True,
        downscale_factor=downscale_factor,
    )
    images.append(img)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for ax, serial, img in zip(axes.flat, upper_serials, images):
    ax.imshow(img)
    ax.set_title(f"cam {serial}")
    ax.axis("off")
plt.tight_layout()
plt.show()


In [ ]:
def image_to_video_tensor(img: np.ndarray) -> torch.Tensor:
    """HWC float [0,1] -> C,T,H,W float [-1,1] with T=1."""
    tensor = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(1)
    return tensor * 2.0 - 1.0


videos = [image_to_video_tensor(img) for img in images]
print("Single-view tensor shape:", videos[0].shape)


In [ ]:
def load_wan_video_vae_module(module_path: Path):
    """
    Load the module; if it fails due to commented blocks, strip top-level
    triple-quote delimiters and reload from source.
    """
    import importlib.util

    spec = importlib.util.spec_from_file_location("wan_video_vae_local", module_path)
    module = importlib.util.module_from_spec(spec)
    try:
        spec.loader.exec_module(module)
        return module
    except Exception as exc:
        print("Standard import failed, applying source patch:", exc)
        src = module_path.read_text()
        # Remove top-level triple-quote lines that comment out class blocks.
        src = re.sub(r'(?m)^"""\s*$', "", src)
        patched = types.ModuleType("wan_video_vae_patched")
        exec(compile(src, str(module_path), "exec"), patched.__dict__)
        return patched


vae_module = load_wan_video_vae_module(
    diffsynth_root / "diffsynth/models/wan_video_vae.py"
)

WanVideoVAE = getattr(vae_module, "WanVideoVAE", None)
MultiViewWanVideoVAE = getattr(vae_module, "MultiViewWanVideoVAE", None)

print("WanVideoVAE available:", WanVideoVAE is not None)
print("MultiViewWanVideoVAE available:", MultiViewWanVideoVAE is not None)


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

use_multiview = MultiViewWanVideoVAE is not None
if use_multiview:
    vae = MultiViewWanVideoVAE(view_in=len(upper_serials)).to(device)
else:
    if WanVideoVAE is None:
        raise RuntimeError("WanVideoVAE is not available in the module.")
    vae = WanVideoVAE().to(device)

# Optional: load a checkpoint if you have one.
# checkpoint_path = Path("/path/to/wan_video_vae.ckpt")
# if checkpoint_path.exists():
#     state = torch.load(checkpoint_path, map_location="cpu")
#     converter = vae.state_dict_converter()
#     vae.load_state_dict(converter.from_civitai(state), strict=False)

vae.eval()
print("Using device:", device)
print("Multi-view mode:", use_multiview)


In [ ]:
with torch.no_grad():
    if use_multiview:
        # [B, V, C, T, H, W]
        video_batch = torch.stack(videos).unsqueeze(0)
        latents = vae.encode(video_batch, device=device)
        recon = vae.decode(latents, device=device)
        recon_videos = recon.squeeze(0)
    else:
        latents = vae.encode(videos, device=device)
        recon_videos = vae.decode(latents, device=device)

recon_images = []
for v in recon_videos:
    img = (v[:, 0].clamp(-1, 1) + 1.0) / 2.0
    recon_images.append(img.permute(1, 2, 0).cpu().numpy())

print("Latent shape:", latents.shape)
print("Recon tensor shape:", recon_videos.shape)


In [ ]:
fig, axes = plt.subplots(len(upper_serials), 2, figsize=(8, 3 * len(upper_serials)))
for i, serial in enumerate(upper_serials):
    axes[i, 0].imshow(images[i])
    axes[i, 0].set_title(f"Original cam {serial}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(recon_images[i])
    axes[i, 1].set_title(f"Reconstruction cam {serial}")
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()
